In [83]:
import os
import re
from pathlib import Path

# --- Configuration des chemins ---
ROOT_DIR = Path.cwd().absolute()
FILES_DIR = ROOT_DIR / "files" / "Anglais"
PAGES_DIR = ROOT_DIR / "pages"
TEMPLATE_PATH = ROOT_DIR / "template.html"

# Cible en_vocab.html directement dans le dossier pages/
EN_VOCAB_ROOT_PATH = PAGES_DIR / "en_vocab.html"

# Créer le dossier pages s'il n'existe pas
PAGES_DIR.mkdir(parents=True, exist_ok=True)

# Charger le modèle HTML
with open(TEMPLATE_PATH, "r", encoding="utf-8") as f:
    TEMPLATE_CONTENT = f.read()

def formater_nom_fichier(chaine):
    """Nettoie une chaîne pour en faire un nom de fichier web propre (slug)"""
    chaine = chaine.lower()
    chaine = re.sub(r'[^\w\s-]', '', chaine)
    return re.sub(r'[-\s]+', '-', chaine).strip('-')

def construire_chemin_page(segments, est_fiche=False, est_racine_vocab=False):
    """Génère le chemin complet du fichier HTML dans PAGES_DIR"""
    if est_racine_vocab:
        return EN_VOCAB_ROOT_PATH
        
    theme_slugs = [formater_nom_fichier(s) for s in segments[:-1]] if est_fiche else [formater_nom_fichier(s) for s in segments]
    
    # Remplacement du tiret par un tiret bas (_) pour marquer le changement de dossier
    nom_base = "en_vocab_theme-" + "_".join(theme_slugs)
    if est_fiche:
        nom_base += f"_liste-{formater_nom_fichier(segments[-1])}"
        
    return PAGES_DIR / f"{nom_base}.html"

def convertir_type(v_type):
    """Transforme les codes de types en libellés propres"""
    v_type = v_type.strip().upper()
    mapping = {
        'N': 'noun',
        'V': 'verb',
        'VI': 'irr. verb',
        'ADJ': 'adj.',
        'ABREV': 'abrev.',
        'A': 'other',
        'DEF': 'notion'
    }
    return mapping.get(v_type, v_type.lower())

def convertir_provenance(prov):
    """Transforme les codes de provenance en libellés propres"""
    prov = prov.strip().upper()
    mapping = {
        'LIVRE': 'Livre',
        'LIVREM': 'Marge du livre',
        'COURS': 'Vu en cours',
        'SUPPTHEME': 'Supplément du thème',
        'SUPPLIVRE': 'Livre (pas vu)',
        'SUPP': 'Supplément'
    }
    return mapping.get(prov, prov)

def generer_breadcrumbs(segments, est_fiche=False, est_racine_vocab=False):
    """Construit le fil d'Ariane adapté à une exécution où toutes les pages sont dans /pages"""
    breadcrumbs = ['<a href="../index.html">Accueil</a>']
    
    if est_racine_vocab:
        breadcrumbs.append('<a href="./en_vocab.html">Vocabulaire</a>')
        return " > ".join(breadcrumbs)
        
    breadcrumbs.append('<a href="./en_vocab.html">Vocabulaire</a>')
    
    for i in range(len(segments)):
        current_segments = segments[:i+1]
        is_last = (i == len(segments) - 1)
        
        if is_last and est_fiche:
            target_file = construire_chemin_page(segments, est_fiche=True).name
        else:
            target_file = construire_chemin_page(current_segments, est_fiche=False).name
            
        titre_segment = segments[i].replace(";", " - ").replace("-", " ").capitalize()
        breadcrumbs.append(f'<a href="./{target_file}">{titre_segment}</a>')
        
    return " > ".join(breadcrumbs)

# --- 1. Cartographie de l'arborescence ---
arborescence = {}
themes_principaux = set()

for root, dirs, files in os.walk(FILES_DIR):
    path_root = Path(root)
    relative_parts = path_root.relative_to(FILES_DIR).parts
    
    if not relative_parts:
        for d in dirs:
            themes_principaux.add(d)
            if (d,) not in arborescence:
                arborescence[(d,)] = {'sous_themes': set(), 'fiches': set()}
        continue

    current_node = relative_parts
    if current_node not in arborescence:
        arborescence[current_node] = {'sous_themes': set(), 'fiches': set()}

    for d in dirs:
        arborescence[current_node]['sous_themes'].add(d)
        child_node = current_node + (d,)
        if child_node not in arborescence:
            arborescence[child_node] = {'sous_themes': set(), 'fiches': set()}
            
    for f in files:
        if f.endswith('.txt'):
            arborescence[current_node]['fiches'].add(f[:-4])

# --- 2. Génération des pages ---

# -- A. Génération de la page racine pages/en_vocab.html --
corps_racine = '<p class="intro-text">Les fiches sont triées par thèmes, elles font en général une quarantaine de mots pour que ça fasse pas trop lourd à apprendre</p>\n'
corps_racine += '<section class="subject-section">\n<h3>Thèmes principaux</h3>\n<div class="topic-grid">\n'
for t in sorted(themes_principaux):
    lien_cible = f"./{construire_chemin_page((t,), est_fiche=False).name}"
    nom_image = f"en_vocab_theme-{formater_nom_fichier(t)}.png"
    
    corps_racine += f"""
    <a href="{lien_cible}" class="subject-topic">
        <span class="topic-title">{t.replace("-", " ").capitalize()}</span>
        <img src="../images/{nom_image}" alt="img {t}" class="topic-image">
    </a>\n"""
corps_racine += '</div>\n</section>\n'

html_racine = TEMPLATE_CONTENT.format(
    PAGE_TITLE="Vocabulaire",
    MAIN_TITLE="ANGLAIS > Vocabulaire",
    BREADCRUMBS=generer_breadcrumbs([], est_racine_vocab=True),
    CONTENT=corps_racine
)
with open(EN_VOCAB_ROOT_PATH, "w", encoding="utf-8") as f_out:
    f_out.write(html_racine)


# -- B. Génération des Fiches de Vocabulaire --
for node, data in arborescence.items():
    for fiche in data['fiches']:
        chemin_txt = FILES_DIR / os.path.join(*node) / f"{fiche}.txt"
        segments_fiche = list(node) + [fiche]
        fichier_html_cible = construire_chemin_page(segments_fiche, est_fiche=True)
        
        titre_propre = fiche.split(';')[0].replace("-", " ").capitalize() if ';' in fiche else fiche.replace("-", " ").capitalize()
        
        tableau_html = '<table class="vocab-table">\n<thead>\n<tr>'
        tableau_html += '<th>Type</th><th>Anglais</th><th>Français</th><th>Niveau</th><th>Note</th><th>Exemple</th><th>Provenance</th>'
        tableau_html += '</tr>\n</thead>\n<tbody>\n'
        
        lien_quizlet = None
        
        with open(chemin_txt, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                
                if line.startswith("QUIZLET;") or line.startswith("LIEN;") or line.startswith("http"):
                    parts_lien = line.split(';')
                    lien_quizlet = parts_lien[1].strip() if len(parts_lien) > 1 else parts_lien[0].strip()
                    continue
                
                if line.startswith("#"):
                    continue
                
                parts = [p.strip() for p in line.split(';')]
                if len(parts) < 7:
                    continue
                
                anglais = parts[0]
                francais = parts[1]
                note = parts[2]
                exemple = parts[3]
                v_type = parts[4]
                provenance = parts[5]
                niveau = parts[6]
                
                type_propre = convertir_type(v_type)
                provenance_propre = convertir_provenance(provenance)
                
                note_propre = "" if note == "..." else note
                exemple_propre = "" if exemple == "..." else exemple
                
                type_classe = formater_nom_fichier(type_propre)
                niveau_classe = niveau.strip().lower()

                cell_type = f"<span class='badge badge-type badge-type-{type_classe}'>{type_propre}</span>" if type_propre != "..." else ""
                cell_anglais = f"<span class='txt-anglais'>{anglais}</span>"
                cell_francais = f"<span class='txt-francais'>{francais}</span>"
                cell_level = f"<span class='badge badge-level badge-level-{niveau_classe}'>{niveau}</span>" if niveau != "..." else ""
                cell_note = f"<span class='txt-note'>{note_propre}</span>"
                cell_exemple = f"<span class='txt-exemple'>{exemple_propre}</span>"
                cell_prov = f"<span class='badge badge-prov'>{provenance_propre}</span>" if provenance_propre != "..." else ""
                
                tableau_html += f"<tr><td>{cell_type}</td><td>{cell_anglais}</td><td>{cell_francais}</td><td>{cell_level}</td><td>{cell_note}</td><td>{cell_exemple}</td><td>{cell_prov}</td></tr>\n"
        
        tableau_html += "</tbody>\n</table>"
        
        html_bouton_quizlet = ""
        if lien_quizlet:
            texte_chemin = " / ".join([s.replace("-", " ").capitalize() for s in node])
            html_bouton_quizlet = f"""
            <a href="{lien_quizlet}" target="_blank" class="quizlet-card">
                <div class="quizlet-icon-wrapper">
                    <img src="../images/quizlet_logo.png" alt="Q" class="quizlet-icon">
                </div>
                <div class="quizlet-text">
                    <div class="quizlet-path">{texte_chemin}</div>
                    <div class="quizlet-title">{titre_propre}</div>
                </div>
            </a>
            """
        
        corps_page = f"""
        <div class="fiche-header-container">
            <div class="fiche-titles">
                <p class="intro-text">Fiche de vocabulaire</p>
                <h3 class="fiche-title">{titre_propre}</h3>
            </div>
            {html_bouton_quizlet}
        </div>
        {tableau_html}
        """
        
        breadcrumbs_html = generer_breadcrumbs(segments_fiche, est_fiche=True)
        html_final = TEMPLATE_CONTENT.format(
            PAGE_TITLE=titre_propre,
            MAIN_TITLE="ANGLAIS > Vocabulaire",
            BREADCRUMBS=breadcrumbs_html,
            CONTENT=corps_page
        )
        
        with open(fichier_html_cible, "w", encoding="utf-8") as f_out:
            f_out.write(html_final)

# -- C. Génération des pages de Navigation --
for node, data in arborescence.items():
    fichier_html_cible = construire_chemin_page(node, est_fiche=False)
    titre_navigation = node[-1].replace("-", " ").capitalize()
    
    corps_page = ""
    
    if data['sous_themes']:
        corps_page += '<section class="subject-section">\n<h3>Sous-thèmes disponibles</h3>\n<div class="topic-grid">\n'
        for st in sorted(data['sous_themes']):
            node_cible = node + (st,)
            lien_cible = f"./{construire_chemin_page(node_cible, est_fiche=False).name}"
            theme_slugs = [formater_nom_fichier(s) for s in node_cible]
            nom_image = f"en_vocab_theme-{'_'.join(theme_slugs)}.png"
            corps_page += f"""<a href="{lien_cible}" class="subject-topic"><span class="topic-title">{st.replace("-", " ").capitalize()}</span><img src="../images/{nom_image}" alt="img {st}" class="topic-image"></a>\n"""
        corps_page += '</div>\n</section>\n'
        
    if data['fiches']:
        corps_page += '<section class="fiches-section">\n<h3>Listes de vocabulaire</h3>\n<ul class="subject-list">\n'
        for fiche in sorted(data['fiches']):
            segments_fiche = list(node) + [fiche]
            lien_cible = f"./{construire_chemin_page(segments_fiche, est_fiche=True).name}"
            titre_fiche = fiche.split(';')[0].replace("-", " ").capitalize() if ';' in fiche else fiche.replace("-", " ").capitalize()
            corps_page += f'<li><a href="{lien_cible}" class="subject-link">📄 {titre_fiche}</a></li>\n'
        corps_page += '</ul>\n</section>\n'
        
    breadcrumbs_html = generer_breadcrumbs(node, est_fiche=False)
    html_final = TEMPLATE_CONTENT.format(
        PAGE_TITLE=titre_navigation,
        MAIN_TITLE="ANGLAIS > Vocabulaire",
        BREADCRUMBS=breadcrumbs_html,
        CONTENT=corps_page
    )
    
    with open(fichier_html_cible, "w", encoding="utf-8") as f_out:
        f_out.write(html_final)

print("Génération réussie avec support des liens Quizlet !")

Génération réussie avec support des liens Quizlet !


In [81]:
import os
import re
from pathlib import Path

# --- Configuration des chemins ---
ROOT_DIR = Path.cwd().absolute()
FILES_DIR = ROOT_DIR / "files" / "Anglais"
OUTPUT_DIR = ROOT_DIR / "texts-lists"

# Créer le dossier de destination s'il n'existe pas
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def formater_nom_fichier(chaine):
    """Nettoie une chaîne pour en faire un nom de fichier propre (slug)"""
    chaine = chaine.lower()
    chaine = re.sub(r'[^\w\s-]', '', chaine)
    return re.sub(r'[-\s]+', '-', chaine).strip('-')

def construire_nom_liste(node, fiche):
    """Génère le nom du fichier .txt cible basé sur l'arborescence (comme les pages)"""
    theme_slugs = [formater_nom_fichier(s) for s in node]
    nom_base = "en_vocab_theme-" + "_".join(theme_slugs)
    nom_base += f"_liste-{formater_nom_fichier(fiche)}"
    return f"{nom_base}.txt"

# --- Parcours de l'arborescence et conversion ---
compteur = 0

for root, dirs, files in os.walk(FILES_DIR):
    path_root = Path(root)
    relative_parts = path_root.relative_to(FILES_DIR).parts
    
    # On traite uniquement les fichiers dans les sous-dossiers thématiques
    if not relative_parts:
        continue

    for f in files:
        if f.endswith('.txt'):
            nom_fiche_sans_ext = f[:-4]
            chemin_txt_origine = path_root / f
            
            # Liste pour stocker les blocs de mots formatés
            blocs_mots = []
            
            with open(chemin_txt_origine, "r", encoding="utf-8") as f_in:
                for line in f_in:
                    line = line.strip()
                    if not line or line.startswith("#"):
                        continue
                    
                    # Ignorer la ligne de lien si elle existe
                    if line.startswith("QUIZLET;") or line.startswith("LIEN;") or line.startswith("http"):
                        continue
                    
                    # Découpage des 7 colonnes standard
                    parts = [p.strip() for p in line.split(';')]
                    if len(parts) < 7:
                        continue
                    
                    anglais = parts[0]
                    francais = parts[1]
                    note = parts[2]
                    exemple = parts[3]
                    
                    # Remplacement des balises HTML <br> par des retours à la ligne
                    anglais = re.sub(r'<br\s*/?>', '\n', anglais)
                    francais = re.sub(r'<br\s*/?>', '\n', francais)
                    
                    # Traitement de l'exemple (Ajout du préfixe conditionnel)
                    if exemple == "...":
                        exemple_final = ""
                    else:
                        exemple_nettoye = re.sub(r'<br\s*/?>', '\n', exemple)
                        exemple_final = f"\n\nExemple : {exemple_nettoye}"
                        
                    # Traitement de la note (Ajout du préfixe conditionnel)
                    if note == "...":
                        note_final = ""
                    else:
                        note_nettoye = re.sub(r'<br\s*/?>', '\n', note)
                        note_final = f"\n\nNote : {note_nettoye}"
                    
                    # Construction du format individuel :
                    # {anglais}\n\nExemple : {exemple}|||{français}\n\nNote : {note}
                    partie_gauche = f"{anglais}{exemple_final}".strip()
                    partie_droite = f"{francais}{note_final}".strip()
                    
                    bloc_formate = f"{partie_gauche}|||{partie_droite}"
                    blocs_mots.append(bloc_formate)
            
            # Si le fichier contenait des mots valides, on l'exporte
            if blocs_mots:
                # Jointure de tous les blocs par "||||||"
                contenu_final = "||||||".join(blocs_mots)
                
                # Détermination du nom approprié basé sur l'arborescence
                nom_fichier_cible = construire_nom_liste(relative_parts, nom_fiche_sans_ext)
                chemin_cible = OUTPUT_DIR / nom_fichier_cible
                
                with open(chemin_cible, "w", encoding="utf-8") as f_out:
                    f_out.write(contenu_final)
                
                print(f"Créé : {nom_fichier_cible}")
                compteur += 1

print(f"\nTerminé ! {compteur} fichiers ont été convertis et sauvegardés dans /texts-lists")

Créé : en_vocab_theme-media_liste-american-press-and-tv-la-presse-et-la-télévision-américaine.txt
Créé : en_vocab_theme-media_liste-british-press-and-tv-la-presse-et-la-télévision-britanique.txt
Créé : en_vocab_theme-media_the-freedom-of-the-media_liste-censorshipla-censure.txt
Créé : en_vocab_theme-media_the-freedom-of-the-media_liste-information-or-muckraking-information-ou-chasse-au-scandale.txt
Créé : en_vocab_theme-media_the-freedom-of-the-media_liste-surveillancela-surveillance.txt
Créé : en_vocab_theme-media_the-press_liste-journalismle-journalisme.txt
Créé : en_vocab_theme-media_the-press_liste-newspaper-and-magazinesjournaux-et-magazines.txt
Créé : en_vocab_theme-politics_war_liste-gunsarmes.txt
Créé : en_vocab_theme-sciences-tech-health_health_diseases_liste-disorders-and-disabilititestroubles-et-handicaps.txt
Créé : en_vocab_theme-sciences-tech-health_health_diseases_liste-infectious-diseasesles-maladies-infectueuses.txt
Créé : en_vocab_theme-sciences-tech-health_health_dise